# Project: Retail Sales Data ETL Pipeline

**Author:** Aditya Zaldy
**Tools:** Python, Pandas, Jupyter  

---

### Executive Summary
This notebook documents the **Extract, Transform, Load (ETL)** process for raw retail sales data. The goal is to clean dirty raw data, handle anomalies (such as invalid customer IDs and formatting issues), and generate a high-quality dataset ready for business intelligence analysis.

## 1. Environment Setup & Data Ingestion
Importing necessary libraries and defining the directory paths. 

> **Technical Note:** When loading the dataset, we explicitly force the `credit_card` column to be read as a **string (object)**. This prevents integer overflow errors and preserves the exact card number sequence.

In [47]:
import pandas as pd
import numpy as np
import os

In [48]:
# Data Path
file_path = '../data/raw/Fact_Sales_1.csv'

# Load Data
df = pd.read_csv(file_path, dtype={'credit_card': str})
df.head()

,transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price
0,1,2021-05-04 02:00:00,P0494,4,visa,4041593010498829,F,17.33,2,18.29
1,2,2021-05-04 03:04:00,P0221,5,visa,4041596151234556,F,0.59,1,1.49
2,3,2021-05-04 03:56:00,P0625,5,visa,4041594885335898,F,5.15,3,5.89
3,4,2021-05-04 05:20:00,P0431,8,mastercard,5108753677552345,F,10.67,2,11.59
4,5,2021-05-04 05:45:00,P0058,5,mastercard,5108752372298261,T,11.38,2,12.39


## 2. Data Profiling & Audit
Before cleaning, we must understand the "health" of the data. We will check for:
* **Missing Values:** specifically in the `payment` column.
* **Anomalies:** Checking min/max values in IDs and financial columns.
* **Data Types:** Ensuring dates and categories are formatted correctly.

In [49]:
# Data Info
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_id      4410 non-null   int64  
 1   transactional_date  4410 non-null   object 
 2   product_id          4410 non-null   object 
 3   customer_id         4410 non-null   int64  
 4   payment             4234 non-null   object 
 5   credit_card         4410 non-null   object 
 6   loyalty_card        4410 non-null   object 
 7   cost                4410 non-null   float64
 8   quantity            4410 non-null   int64  
 9   price               4410 non-null   float64
dtypes: float64(2), int64(3), object(5)
memory usage: 344.7+ KB


,transaction_id,customer_id,cost,quantity,price
count,4410.000000,4410.000000,4410.000000,4410.000000,4410.000000
mean,2205.500000,5.980499,10.153653,2.110884,11.334422
std,1273.201673,2.001152,5.733201,1.253417,6.038442
min,1.000000,-1.000000,0.090000,1.000000,1.090000
25%,1103.250000,5.000000,5.210000,1.000000,5.990000
50%,2205.500000,6.000000,10.240000,2.000000,11.490000
75%,3307.750000,7.000000,15.320000,3.000000,17.090000
max,4410.000000,13.000000,20.000000,10.000000,21.490000


In [50]:
# Check 'customer_id' value counts
df['customer_id'].value_counts()

customer_id
 6     869
 7     797
 5     770
 4     541
 8     514
 9     294
 3     290
 2     120
 10    117
 1      42
 11     28
 12     13
 0      10
-1       3
 13      2
Name: count, dtype: int64

## 3. Data Cleaning & Transformation
Based on the audit findings, we execute the following cleaning steps:

### 3.1 Handling Anomalies
* **Issue:** Found `customer_id` with value `-1`.
* **Action:** Rows with invalid IDs are treated as system errors/garbage data and will be **dropped** to maintain customer analytics accuracy.

### 3.2 Formatting
* **Missing Payments:** `NaN` values in the `payment` column are filled with `'Unknown'`.
* **Text Normalization:** Payment methods are converted to **Title Case** (e.g., 'visa' $\rightarrow$ 'Visa') for consistency.
* **Date Parsing:** Converting `transactional_date` to datetime objects.

In [51]:
# Delete Invalid Customer Id
print(f"row count : {len(df)}")
df = df[df['customer_id'] != -1]
print(f"row count : {len(df)}")

row count : 4410
row count : 4407


In [52]:
# Check Nulls in 'payment' column
print("Nulls in Payment :", df['payment'].isnull().sum())

df['payment'] = df['payment'].fillna('Unknown')

print("Nulls in Payment :", df['payment'].isnull().sum())

Nulls in Payment : 176
Nulls in Payment : 0


In [53]:
# Convert 'transactional_date' to datetime
df['transactional_date'] = pd.to_datetime(df['transactional_date'])
# Standardize payment method names
df['payment'] = df['payment'].str.title()

df.head()

,transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price
0,1,2021-05-04 02:00:00,P0494,4,Visa,4041593010498829,F,17.33,2,18.29
1,2,2021-05-04 03:04:00,P0221,5,Visa,4041596151234556,F,0.59,1,1.49
2,3,2021-05-04 03:56:00,P0625,5,Visa,4041594885335898,F,5.15,3,5.89
3,4,2021-05-04 05:20:00,P0431,8,Mastercard,5108753677552345,F,10.67,2,11.59
4,5,2021-05-04 05:45:00,P0058,5,Mastercard,5108752372298261,T,11.38,2,12.39


## 4. Feature Engineering
To enrich the dataset for the BI team, we calculate the following business metrics:
* **Total Amount:** `Quantity` * `Price`
* **Total Cost:** `Quantity` * `Cost`
* **Net Profit:** `Total Amount` - `Total Cost`

In [54]:
# Calculate total amount, total cost, and net profit
df['total_amount'] = df['quantity'] * df['price']
df['total_cost'] = df['quantity'] * df['cost']
df['net_profit'] = df['total_amount'] - df['total_cost']

df[['quantity', 'price', 'cost', 'total_amount', 'total_cost', 'net_profit']].head()

,quantity,price,cost,total_amount,total_cost,net_profit
0,2,18.29,17.33,36.58,34.66,1.92
1,1,1.49,0.59,1.49,0.59,0.90
2,3,5.89,5.15,17.67,15.45,2.22
3,2,11.59,10.67,23.18,21.34,1.84
4,2,12.39,11.38,24.78,22.76,2.02


## 5. Post-Cleaning Quality Assurance (QA)
Ensure the cleaning rules were applied and no new errors were introduced.

**Checklist:**
1.  ✅ No duplicate Transaction IDs.
2.  ✅ No negative values in price/quantity columns.
3.  ✅ No null values in critical columns.

In [55]:
# Data Validation Checks
if df['transaction_id'].duplicated().any():
    print("Duplicates found in transaction_id")
else:
    print("No duplicates in transaction_id")
    
# Check for negative values in quantity, price, cost, total_amount
cols_to_check = ['quantity', 'price', 'cost', 'total_amount']
if (df[cols_to_check] < 0).any().any():
    print("\nNegative values found in columns:", cols_to_check)
else:
    print("\nNo negative values in columns:", cols_to_check)

print("\nUnique payment methods:", df['payment'].unique())

No duplicates in transaction_id

No negative values in columns: ['quantity', 'price', 'cost', 'total_amount']

Unique payment methods: ['Visa' 'Mastercard' 'Americanexpress' 'Unknown']


## 6. Final Data Load
The cleaned dataset is verified and saved to the processed directory.

* **Output File:** `data/clean/Sales_Cleaned.csv`

In [57]:
# Save Cleaned Data
output_file = '../data/clean/Sales_Cleaned.csv'

os.makedirs(os.path.dirname(output_file), exist_ok=True)

df.to_csv(output_file, index=False)
print(f"Cleaned data saved to {output_file}")

Cleaned data saved to ../data/clean/Sales_Cleaned.csv
